# 01 — Dos arquivos baixados às tabelas da pesquisa

**Pergunta desta etapa:** quais estabelecimentos e quais regiões entram na pesquisa, e como ligamos seus dados?

Vamos trabalhar com duas tabelas diferentes:

- **Pronto-atendimentos:** uma linha por estabelecimento; identificador `CO_CNES`.
- **Setores:** uma linha por código de setor do IBGE; identificador `CD_SETOR`.

Um setor é uma região do mapa. Ele não é um morador nem um estabelecimento. População, área e renda são informações dessa região.

Este notebook lê os arquivos já baixados e **refaz o preparo**, inclusive a conferência histórica de coordenadas a partir das respostas de geocodificação guardadas em cache. Não chama os scripts de análise nem consulta serviços externos. Os resultados ficam separados em `dados/processados/retomada/base/`.

As regras de recorte, coordenadas e filtros são as já usadas e aprovadas. Se a reprodução divergir do cadastro de referência, a execução para antes de adotar os resultados. A comparação com os arquivos anteriores é uma conferência, não sua utilização como fonte das novas tabelas.


In [1]:
from pathlib import Path
import json
import re
import unicodedata

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/"PLANO.md").exists())
DADOS = RAIZ/"dados"
SAIDA = DADOS/"processados/retomada/base"
SAIDA.mkdir(parents=True,exist_ok=True)
GPKG_DIR = DADOS/"ibge/setores_gpkg"
CAPITAIS = [
    "Porto Velho - RO","Rio Branco - AC","Manaus - AM","Boa Vista - RR","Belém - PA",
    "Macapá - AP","Palmas - TO","São Luís - MA","Teresina - PI","Fortaleza - CE",
    "Natal - RN","João Pessoa - PB","Recife - PE","Maceió - AL","Aracaju - SE",
    "Salvador - BA","Belo Horizonte - MG","Vitória - ES","Rio de Janeiro - RJ",
    "São Paulo - SP","Curitiba - PR","Florianópolis - SC","Porto Alegre - RS",
    "Campo Grande - MS","Cuiabá - MT","Goiânia - GO","Brasília - DF",
]
print("Entrada: arquivos locais em",DADOS)
print("Saída separada:",SAIDA.relative_to(RAIZ))


Entrada: arquivos locais em /home/liasucf/ciencia-de-cidades/dados
Saída separada: dados/processados/retomada/base


## 1. Identificar as capitais e suas populações

O arquivo do IBGE informa o código municipal, o nome e a população de 2022. Selecionamos as 27 capitais pelos nomes que já estavam definidos no projeto.

**Por que aparecem dois códigos municipais?** O arquivo do IBGE usa sete dígitos e o CNES usa seis neste conjunto. Guardamos ambos para cruzar as tabelas; não cruzamos cidades apenas por nomes escritos de formas diferentes.


In [2]:
resposta_pop = json.loads((DADOS/"ibge/pop2022.json").read_text())
series = resposta_pop[0]["resultados"][0]["series"]
populacao = pd.DataFrame([{
    "co_ibge7": str(s["localidade"]["id"]),
    "municipio": s["localidade"]["nome"],
    "populacao_2022": int(s["serie"]["2022"])
} for s in series])
capitais = populacao.loc[populacao.municipio.isin(CAPITAIS)].copy()
assert len(capitais)==27 and not capitais.municipio.duplicated().any()
capitais["co_ibge6"] = capitais.co_ibge7.str[:6]
display(capitais.head())


,co_ibge7,municipio,populacao_2022,co_ibge6
16,1100205,Porto Velho - RO,460434,110020
66,1200401,Rio Branco - AC,364756,120040
111,1302603,Manaus - AM,2063689,130260
138,1400100,Boa Vista - RR,413486,140010
169,1501402,Belém - PA,1303403,150140


## 2. Selecionar os estabelecimentos

O critério principal é **tipo 73 do CNES**, com campo de motivo de desativação vazio no arquivo. O nome não precisa conter “UPA”.

A coluna `atende_sus` vem do indicador `CO_AMBULATORIAL_SUS`. Ela define o recorte SUS utilizado; não reclassificamos estabelecimentos pelo nome ou por sua esfera administrativa.

Vamos comparar as contagens geradas com as anteriores para detectar mudanças involuntárias de recorte.


In [3]:
colunas_cnes = [
    "CO_CNES","CO_IBGE","TP_UNIDADE","CO_MOTIVO_DESAB","NO_FANTASIA","NO_RAZAO_SOCIAL",
    "CO_AMBULATORIAL_SUS","DS_ESFERA_ADMINISTRATIVA","DS_TURNO_ATENDIMENTO",
    "NO_LOGRADOURO","NU_ENDERECO","NO_BAIRRO","CO_CEP","NU_LATITUDE","NU_LONGITUDE"
]
cnes = pd.read_csv(DADOS/"cnes/cnes_estabelecimentos.csv",sep=";",encoding="latin1",
                   dtype=str,usecols=colunas_cnes)
selecao = (cnes.TP_UNIDADE=="73") & cnes.CO_MOTIVO_DESAB.isna()
pa = cnes.loc[selecao & cnes.CO_IBGE.isin(capitais.co_ibge6)].copy()
pa = pa.merge(capitais[["co_ibge6","municipio"]],left_on="CO_IBGE",right_on="co_ibge6",validate="many_to_one")
pa["atende_sus"] = pa.CO_AMBULATORIAL_SUS=="SIM"
pa["24h"] = pa.DS_TURNO_ATENDIMENTO.str.startswith("ATENDIMENTO CONTINUO DE 24 HORAS",na=False)
nome = pa.NO_FANTASIA.fillna("")+" | "+pa.NO_RAZAO_SOCIAL.fillna("")
pa["nome_tem_upa"] = nome.str.contains(r"\bUPA\b|UPA ?24|UNIDADE DE PRONTO ATENDIMENTO|UNIDADE PRONTO ATENDIMENTO",regex=True)
pa["lat"] = pd.to_numeric(pa.NU_LATITUDE,errors="coerce")
pa["lon"] = pd.to_numeric(pa.NU_LONGITUDE,errors="coerce")
pa["tem_coordenada"] = pa.lat.notna() & pa.lon.notna() & (pa.lat!=0)
assert not pa.CO_CNES.duplicated().any()
print("Estabelecimentos selecionados:",len(pa),"| SUS:",int(pa.atende_sus.sum()))
display(pa[["CO_CNES","municipio","NO_FANTASIA","atende_sus"]].head())
del cnes


Estabelecimentos selecionados: 274 | SUS: 245


,CO_CNES,municipio,NO_FANTASIA,atende_sus
0,3913,Salvador - BA,UNIDADE EMERGENCIA GERCINO COELHO,True
1,4340,Salvador - BA,UPA 24H DR HELIO MACHADO,True
2,4774,Salvador - BA,UPA 24H PROF ADROALDO ALBERGARIA,True
3,7587,Salvador - BA,UNIDADE DE EMERGENCIA MAE HILDA JITOLU,True
4,10049,Campo Grande - MS,SESAU UPA CORONEL ANTONINO DR WALFRIDO ARRUDA,True


In [4]:
g = pa.groupby("municipio")
resumo = capitais.set_index("municipio")[["co_ibge7","populacao_2022"]].join([
    g.size().rename("n_pa"),g.atende_sus.sum().rename("n_pa_sus"),
    g["24h"].sum().rename("n_pa_24h"),g.nome_tem_upa.sum().rename("n_nome_upa"),
    g.tem_coordenada.sum().rename("n_com_coord")
]).fillna(0)
contagens = ["n_pa","n_pa_sus","n_pa_24h","n_nome_upa","n_com_coord"]
resumo[contagens] = resumo[contagens].astype(int)
resumo = resumo.sort_values("populacao_2022",ascending=False)
referencia_cap = pd.read_csv(DADOS/"processados/capitais_resumo.csv").set_index("municipio")
pd.testing.assert_frame_equal(
    resumo[contagens].sort_index(),referencia_cap[contagens].sort_index(),check_dtype=False
)
resumo.to_csv(SAIDA/"capitais_resumo.csv")
display(resumo[["populacao_2022","n_pa_sus","n_pa"]])
print("Conferência: contagens reproduzidas.")


,populacao_2022,n_pa_sus,n_pa
municipio,,,
São Paulo - SP,11451999,56,64
Rio de Janeiro - RJ,6211223,31,38
Brasília - DF,2817381,13,14
Fortaleza - CE,2428708,12,13
Salvador - BA,2417678,20,21
Belo Horizonte - MG,2315560,9,11
Manaus - AM,2063689,10,11
Curitiba - PR,1773718,9,9
Recife - PE,1488920,5,6


Conferência: contagens reproduzidas.


## 3. Reproduzir a conferência de coordenadas sem novas consultas

Uma latitude/longitude preenchida pode estar errada. O procedimento anterior comparou as coordenadas com os bairros esperados e com consultas por endereço, CEP ou nome.

Aqui repetimos **as mesmas regras**, mas a consulta lê somente o cache local de respostas. Não estamos escolhendo novos critérios nem substituindo coordenadas por novos resultados da internet.

| Regra histórica | Valor |
|---|---:|
| Bairro considerado próximo | até 1.500 m |
| Tolerância de ponto fora do município | 200 m |
| Confirmação por consulta próxima à coordenada | 500 m; 1.000 m quando falta área esperada |
| Coordenadas compartilhadas entre bairros diferentes | menos de 100 m |
| Erro considerado grosseiro em relação ao bairro | mais de 5.000 m |

O código auxiliar abaixo preserva a lógica do processamento anterior. Ao final, conferimos código por código as coordenadas, os métodos e os estados contra o cadastro já aprovado. Os registros que precisam de revisão manual continuam marcados; não são resolvidos silenciosamente.


In [5]:
LIMITE_PERTO_M = 1500
LIMITE_FORA_M = 200
LIMITE_CONFIRMA_M = 500
LIMITE_COMPARTILHADA_M = 100
LIMITE_GROSSEIRO_M = 5000
BAIRROS_GENERICOS = {frozenset({"CENTRO"})}
ABREVIACOES = {"CONJ":"CONJUNTO","JD":"JARDIM","JARD":"JARDIM","PQ":"PARQUE","VL":"VILA",
               "STA":"SANTA","STO":"SANTO","N":"NOSSA","SRA":"SENHORA","RES":"RESIDENCIAL",
               "NUC":"NUCLEO","HAB":"HABITACIONAL","SETOR":"SETOR","CJ":"CONJUNTO","LUIZ":"LUIS"}
IGNORAR = {"I","II","III","IV","V","BAIRRO","DE","DA","DO","DOS","DAS","E"}
cache = json.loads((DADOS/"processados/geocodificacao_cache.json").read_text())
consultas_ausentes = []

def geocodificar(endereco, cache):
    """Somente consulta local ao cache; nenhuma requisição de rede."""
    chave = json.dumps(endereco,sort_keys=True,ensure_ascii=False)
    if chave not in cache:
        consultas_ausentes.append(chave)
        return None
    return cache[chave]


In [6]:
def tokens(nome):
    if not isinstance(nome, str):
        return frozenset()
    s = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode().upper()
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    return frozenset(ABREVIACOES.get(t, t) for t in s.split() if t not in IGNORAR)

def mesmo_bairro(a, b):
    ta, tb = tokens(a), tokens(b)
    return bool(ta) and bool(tb) and (ta <= tb or tb <= ta)

### Como a escolha da coordenada é feita

A função seguinte tem quatro partes, identificadas pelos comentários:

1. Preparar limites e bairros na mesma projeção em metros.
2. Identificar coordenadas fora do município, longe do bairro ou compartilhadas.
3. Examinar as respostas já coletadas, preservando a coordenada ou aplicando a correção prevista.
4. Registrar o método, o deslocamento e os pontos original/final.

Ela está escrita integralmente no notebook, em vez de ser importada de um script. Esta é a parte mais longa do preparo; a tabela de saída permite examinar cada decisão. Pontos aproximados por bairro não equivalem à localização exata de uma entrada de atendimento.


In [7]:
def conferir_capital(capital, cd_mun, pa, cache):
    uf = capital.split(" - ")[1]
    gpkg = GPKG_DIR / f"{uf}_setores_CD2022.gpkg"
    s = gpd.read_file(gpkg, where=f"CD_MUN='{cd_mun}'", columns=["CD_SETOR", "NM_BAIRRO", "NM_SUBDIST", "NM_DIST"])
    crs_m = s.estimate_utm_crs()
    s = s.to_crs(crs_m)
    municipio = s.union_all()
    # Sem bairros na malha (São Paulo, Brasília, Goiânia...), usa a divisão interna que existir
    camada = next((c for c in ["NM_BAIRRO", "NM_SUBDIST", "NM_DIST"] if s[c].nunique() > 1), None)
    if camada is None:
        bairros = gpd.GeoDataFrame({"NM_BAIRRO": []}, geometry=[], crs=crs_m)
    else:
        bairros = (s.loc[s[camada].notna(), [camada, "geometry"]].dissolve(camada).reset_index()
                   .rename(columns={camada: "NM_BAIRRO"}))
    cidade, estado = capital.split(" - ")

    bairros["tokens"] = [tokens(b) for b in bairros.NM_BAIRRO]

    pontos = gpd.GeoSeries(gpd.points_from_xy(pa.lon, pa.lat), crs="EPSG:4326", index=pa.index).to_crs(crs_m)
    compartilhada = {
        i for i in pa.index for j in pa.index
        if i != j and pontos[i].distance(pontos[j]) < LIMITE_COMPARTILHADA_M
        and not mesmo_bairro(pa.at[i, "NO_BAIRRO"], pa.at[j, "NO_BAIRRO"])
    }

    linhas = []
    for i, u in pa.iterrows():
        p = pontos[i]
        contem = bairros[bairros.contains(p)]
        bairro_coord = contem.NM_BAIRRO.iloc[0] if len(contem) else None
        nome = tokens(u.NO_FANTASIA)
        do_endereco = pd.Series([mesmo_bairro(b, u.NO_BAIRRO) for b in bairros.NM_BAIRRO],
                                index=bairros.index, dtype=bool)
        do_nome = pd.Series([bool(t) and t <= nome and t not in BAIRROS_GENERICOS for t in bairros.tokens],
                            index=bairros.index, dtype=bool)
        alvo = bairros[do_endereco | do_nome]
        bairro_nome = ", ".join(bairros.NM_BAIRRO[do_nome]) or None
        dist_bairro = alvo.distance(p).min() if len(alvo) else None
        fora = municipio.distance(p)

        if fora > LIMITE_FORA_M:
            status = "fora_do_municipio"
        elif i in compartilhada:
            status = "coordenada_compartilhada"
        elif dist_bairro is None:
            status = "sem_area_esperada"
        elif dist_bairro == 0:
            status = "ok"
        elif dist_bairro <= LIMITE_PERTO_M:
            status = "perto"
        else:
            status = "suspeita"

        lat, lon, metodo = u.lat, u.lon, "original"
        if status in {"suspeita", "fora_do_municipio", "coordenada_compartilhada", "sem_area_esperada"}:
            # Sem área esperada, o endereço é a única referência: confirma com tolerância maior
            tolerancia = LIMITE_CONFIRMA_M if status != "sem_area_esperada" else 2 * LIMITE_CONFIRMA_M
            pode_confirmar = status in {"suspeita", "sem_area_esperada"}  # nos outros, a original já é errada
            metodo = "revisar_manual"
            numero = "" if str(u.NU_ENDERECO).upper() in {"S/N", "SN", "NAN"} else f"{u.NU_ENDERECO} "
            consultas = [
                ("rua", {"street": f"{numero}{u.NO_LOGRADOURO}", "city": cidade, "state": estado, "country": "Brasil"}),
                ("cep", {"postalcode": str(u.CO_CEP), "city": cidade, "country": "Brasil"}),
                ("nome", {"q": f"{u.NO_FANTASIA}, {cidade}, {estado}, Brasil"}),
            ]
            for fonte, consulta in consultas:
                g = geocodificar(consulta, cache)
                if not g:
                    continue
                q = gpd.GeoSeries(gpd.points_from_xy([g[1]], [g[0]]), crs="EPSG:4326").to_crs(crs_m).iloc[0]
                if pode_confirmar and q.distance(p) <= tolerancia:
                    metodo = f"original_confirmada_{fonte}"
                    break
                perto_do_bairro = len(alvo) > 0 and alvo.distance(q).min() <= LIMITE_PERTO_M
                if municipio.distance(q) <= LIMITE_FORA_M and perto_do_bairro:
                    lat, lon = g
                    metodo = f"nominatim_{fonte}"
                    break
            grosseiro = (status in {"fora_do_municipio", "coordenada_compartilhada"}
                         or (dist_bairro is not None and dist_bairro > LIMITE_GROSSEIRO_M))
            if metodo == "revisar_manual" and grosseiro and len(alvo):
                # Prefere o bairro do endereço; senão, o bairro do nome
                base = alvo[do_endereco[alvo.index]] if do_endereco[alvo.index].any() else alvo
                c = gpd.GeoSeries([base.geometry.iloc[0].representative_point()], crs=crs_m).to_crs("EPSG:4326")
                lat, lon, metodo = c.y.iloc[0], c.x.iloc[0], "aproximada_pelo_bairro"
            elif metodo == "revisar_manual" and (grosseiro or status == "sem_area_esperada"):
                # Sem o bairro na malha do IBGE: procura o bairro do endereço no OpenStreetMap.
                # Em sem_area_esperada, só move se o bairro estiver a mais de 5 km (erro grosseiro).
                g = geocodificar({"q": f"{u.NO_BAIRRO}, {cidade}, {estado}, Brasil"}, cache)
                if g:
                    q = gpd.GeoSeries(gpd.points_from_xy([g[1]], [g[0]]), crs="EPSG:4326").to_crs(crs_m).iloc[0]
                    minimo = LIMITE_COMPARTILHADA_M if grosseiro else LIMITE_GROSSEIRO_M
                    if municipio.distance(q) <= LIMITE_FORA_M and q.distance(p) > minimo:
                        lat, lon, metodo = g[0], g[1], "aproximada_pelo_bairro_osm"

        movido = gpd.GeoSeries(gpd.points_from_xy([lon], [lat]), crs="EPSG:4326").to_crs(crs_m).distance(p).iloc[0]
        linhas.append({"CO_CNES": u.CO_CNES, "municipio": capital, "NO_FANTASIA": u.NO_FANTASIA,
                       "endereco": f"{u.NO_LOGRADOURO}, {u.NU_ENDERECO}, CEP {u.CO_CEP}",
                       "bairro_cnes": u.NO_BAIRRO, "bairro_no_nome": bairro_nome,
                       "bairro_da_coordenada": bairro_coord,
                       "dist_ao_bairro_esperado_m": None if dist_bairro is None else round(dist_bairro),
                       "fora_do_municipio_m": round(fora), "status": status, "metodo": metodo,
                       "lat": lat, "lon": lon, "deslocamento_m": round(movido),
                       "mapa_original": f"https://www.openstreetmap.org/?mlat={u.lat}&mlon={u.lon}#map=17/{u.lat}/{u.lon}",
                       "mapa_final": f"https://www.openstreetmap.org/?mlat={lat}&mlon={lon}#map=17/{lat}/{lon}"})
    return linhas

In [8]:
colunas_pa = ["CO_CNES","municipio","NO_FANTASIA","DS_ESFERA_ADMINISTRATIVA","DS_TURNO_ATENDIMENTO",
              "atende_sus","24h","nome_tem_upa","NO_LOGRADOURO","NU_ENDERECO","NO_BAIRRO","CO_CEP",
              "NU_LATITUDE","NU_LONGITUDE","lat","lon","tem_coordenada"]
pa = pa[colunas_pa].copy()
pa.to_csv(SAIDA/"pa_capitais.csv",index=False)
linhas = []
for capital, grupo in pa.groupby("municipio"):
    cd_mun = capitais.set_index("municipio").loc[capital,"co_ibge7"]
    linhas.extend(conferir_capital(capital,cd_mun,grupo,cache))
    print(capital,":",len(grupo),"coordenadas conferidas no cache.",flush=True)
conferencia = pd.DataFrame(linhas)
manuais = pd.read_csv(DADOS/"correcoes_manuais.csv",dtype={"CO_CNES":str}).set_index("CO_CNES")
if len(manuais):
    sel = conferencia.CO_CNES.isin(manuais.index)
    conferencia.loc[sel,"lat"] = conferencia.loc[sel,"CO_CNES"].map(manuais.lat)
    conferencia.loc[sel,"lon"] = conferencia.loc[sel,"CO_CNES"].map(manuais.lon)
    conferencia.loc[sel,"metodo"] = "manual"
assert not consultas_ausentes, f"{len(consultas_ausentes)} consultas faltam no cache: discutir antes de adotar coordenadas."
corrigido = pa.drop(columns=["lat","lon"]).merge(
    conferencia[["CO_CNES","lat","lon","status","metodo"]],on="CO_CNES",validate="one_to_one"
)
ref = pd.read_csv(DADOS/"processados/pa_capitais_corrigido.csv",dtype={"CO_CNES":str}).set_index("CO_CNES")
nova = corrigido.set_index("CO_CNES").loc[ref.index]
np.testing.assert_allclose(nova[["lat","lon"]],ref[["lat","lon"]],rtol=0,atol=1e-9)
pd.testing.assert_frame_equal(nova[["status","metodo"]],ref[["status","metodo"]],check_dtype=False)
conferencia.to_csv(SAIDA/"conferencia_coordenadas.csv",index=False)
corrigido.to_csv(SAIDA/"pa_capitais_corrigido.csv",index=False)
display(conferencia.metodo.value_counts().rename("n_unidades").to_frame())
print("Coordenadas e decisões históricas reproduzidas, sem consultas externas.")


Belo Horizonte - MG : 11 coordenadas conferidas no cache.


Belém - PA : 5 coordenadas conferidas no cache.


Boa Vista - RR : 1 coordenadas conferidas no cache.


Brasília - DF : 14 coordenadas conferidas no cache.


Campo Grande - MS : 10 coordenadas conferidas no cache.


Cuiabá - MT : 5 coordenadas conferidas no cache.


... (18 outputs left out)

Teresina - PI : 3 coordenadas conferidas no cache.


Vitória - ES : 3 coordenadas conferidas no cache.


,n_unidades
metodo,
original,187
original_confirmada_rua,24
revisar_manual,20
nominatim_rua,16
aproximada_pelo_bairro,13
aproximada_pelo_bairro_osm,9
nominatim_nome,3
original_confirmada_cep,1
original_confirmada_nome,1


Coordenadas e decisões históricas reproduzidas, sem consultas externas.


## 4. Preparar a tabela dos setores e ligar a renda

O código do setor funciona como uma chave: ligamos a tabela de população/área à tabela de renda pelo `CD_SETOR`.

A renda escolhida no projeto é **V06004**, média do responsável. Mantemos V06006, mediana, apenas para registro. Não chamamos essas medidas de renda per capita.

Os polígonos serão lidos no notebook de distâncias. Se um código tiver várias partes geométricas, sua população e área oficial devem entrar uma única vez.


In [9]:
colunas_setor = ["CD_SETOR","CD_MUN","NM_MUN","NM_UF","SITUACAO","CD_SIT","CD_TIPO",
                  "AREA_KM2","NM_CONCURB","v0001"]
br = pd.read_csv(DADOS/"ibge/censo2022/BR_setores_CD2022.csv",dtype=str,
                 encoding="utf-8-sig",usecols=colunas_setor)
setores = br.loc[br.CD_MUN.isin(capitais.co_ibge7)].copy().rename(columns={"v0001":"populacao"})
del br
for coluna in ["populacao","AREA_KM2"]:
    setores[coluna] = pd.to_numeric(setores[coluna],errors="coerce")
renda = pd.read_csv(DADOS/"ibge/censo2022/Agregados_por_setores_renda_responsavel_BR.csv",
                    sep=";",encoding="latin1",dtype=str,usecols=["CD_SETOR","V06004","V06006"],
                    na_values=["X","."])
renda = renda.loc[renda.CD_SETOR.isin(setores.CD_SETOR)].rename(
    columns={"V06004":"renda_media","V06006":"renda_mediana"}
)
for coluna in ["renda_media","renda_mediana"]:
    renda[coluna] = pd.to_numeric(renda[coluna],errors="coerce")
assert not setores.CD_SETOR.duplicated().any() and not renda.CD_SETOR.duplicated().any()
setores = setores.merge(renda,on="CD_SETOR",how="left",validate="one_to_one")
setores["densidade"] = setores.populacao/setores.AREA_KM2
setores["acima_limiar"] = setores.densidade>=1500
# Conservamos a informação cadastral, sem desenvolver a comparação adiada.
setores["favela"] = setores.CD_TIPO=="1"
setores.to_csv(SAIDA/"setores_capitais.csv",index=False)
print("Códigos de setor:",len(setores),"| com moradores:",int((setores.populacao>0).sum()))


Códigos de setor: 93996 | com moradores: 92494


## 5. Ler uma linha e conferir a conta da densidade

A densidade populacional é **população dividida pela área**. Um setor pode ter poucos moradores e ser denso se sua área for pequena. Por isso não podemos substituir densidade por população.

A separação de 1.500 moradores/km² é somente um corte descritivo aplicado a setores. Não estamos reproduzindo o método completo de manchas urbanas em grade de 1 km² nem implementando CCA.


In [10]:
exemplo = setores.loc[(setores.NM_MUN=="Fortaleza") & (setores.populacao>0)].head(3).copy()
exemplo["conta_pop_div_area"] = exemplo.populacao/exemplo.AREA_KM2
display(exemplo[["CD_SETOR","populacao","AREA_KM2","conta_pop_div_area","renda_media"]].round(3))
ref_setores = pd.read_csv(DADOS/"processados/setores_capitais.csv",dtype={"CD_SETOR":str}).set_index("CD_SETOR")
nova = setores.set_index("CD_SETOR").loc[ref_setores.index]
for coluna in ["populacao","AREA_KM2","renda_media","renda_mediana","densidade"]:
    np.testing.assert_allclose(nova[coluna],ref_setores[coluna],rtol=1e-12,equal_nan=True)
print("População, área, renda e densidade reproduzidas a partir dos arquivos baixados.")
display(Markdown(
    "**O que sai desta etapa:** cadastro das capitais, unidades selecionadas, coordenadas "
    "reproduzidas e uma tabela de atributos por setor. Ainda não calculamos aqui a distância "
    "de cada setor. O próximo passo didático é [02_dijkstra_exemplo.ipynb](02_dijkstra_exemplo.ipynb); "
    "o cálculo de todos os setores está em [03_distancias_completas.ipynb](03_distancias_completas.ipynb)."
))


,CD_SETOR,populacao,AREA_KM2,conta_pop_div_area,renda_media
12133,230440005130001,866,0.044,19473.651,1474.07
12134,230440005130002,35,0.187,187.349,1664.47
12135,230440005130003,125,0.317,394.882,3315.34


População, área, renda e densidade reproduzidas a partir dos arquivos baixados.


**O que sai desta etapa:** cadastro das capitais, unidades selecionadas, coordenadas reproduzidas e uma tabela de atributos por setor. Ainda não calculamos aqui a distância de cada setor. O próximo passo didático é [02_dijkstra_exemplo.ipynb](02_dijkstra_exemplo.ipynb); o cálculo de todos os setores está em [03_distancias_completas.ipynb](03_distancias_completas.ipynb).